# STRATA: voltar ao BF16 decodificando enquanto baixa (Llama-3.1-8B, Colab)

Quem já tem o Q4_K_M baixa só os arquivos de estrato (6,25 GB no 8B) para ter o BF16 exato. Antes, era preciso baixar tudo e só depois rodar o `junta`. Agora:

- `baixa_estratos.py` pede os arquivos por faixas HTTP, na ordem em que o `junta` os lê, e grava cada um em ordem;
- `strata-ref junta --espera` decodifica cada tensor assim que os bytes dele chegam, conferindo o hash de cada pedaço.

Os arquivos de estrato não estão no Hub, então a "rede" é um servidor local com teto de banda (100, 300 e 1000 Mbit/s e sem teto). O notebook compara três jeitos de chegar ao BF16 exato: **em fluxo**, **baixar e depois juntar**, e **baixar o BF16 cru** na mesma banda medida. No contêiner, com o 1B, a 100 Mbit/s deu 75 s em fluxo, 98 s baixando e depois juntando, e 199 s para o BF16 cru.

**Runtime:** CPU com RAM alta (o mesmo do 70B). Leva ~40 min e usa ~35 GB de disco. No fim, baixa um zip; mande ele na thread.

In [ ]:
# 1. Máquina, Rust e o strata-ref (≈ 5 min)
import os, shutil, subprocess, sys
def sh(cmd, mostra=True):
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, executable="/bin/bash")
    if mostra:
        print(r.stdout[-3000:])
    if r.returncode:
        raise SystemExit(f"falhou: {cmd}")
    return r.stdout
PASTA = "/content/strata_fluxo"
os.makedirs(PASTA, exist_ok=True)
print(sh("lscpu | grep -E 'Model name|^CPU\\(s\\)'", False), "sha_ni:", "sha_ni" in open("/proc/cpuinfo").read())
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
REPO = "/content/strata"
if not os.path.exists(REPO):
    sh(f"git clone -q --depth 1 -b claude/project-thread-btiyii https://github.com/cantyll/strata {REPO}")
sh(f"cd {REPO} && git log -1 --oneline && cd leitor-rs && cargo build --release -q 2>&1 | tail -3; ls -la target/release/strata-ref")
sh("pip install -q -U huggingface_hub hf_xet psutil", False)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

In [ ]:
# 2. Baixa o 8B e grava os arquivos de estrato (o mesmo teste de antes; ≈ 15 min)
cmd = [sys.executable, "-u", "tests/dist/teste_grande.py", "--hf", "unsloth/Meta-Llama-3.1-8B-Instruct",
       "--gguf-repo", "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", "--gguf", "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf", "--dir", PASTA]
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end="")
p.wait()
assert p.returncode == 0, "o teste falhou"

In [ ]:
# 3. Em fluxo × baixa e depois junta × BF16 cru, a 100/300/1000 Mbit/s e sem teto (≈ 20 min)
import json
BF16 = json.load(open(f"{PASTA}/resultados/resultado.json"))["bf16"]
cmd = [sys.executable, "-u", "tests/dist/fluxo.py", "--dist", f"{PASTA}/dist", "--gguf", f"{PASTA}/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
       "--bf16-bytes", str(BF16), "--mbit", "100,300,1000,0", "--tmp", "/content/fluxo_tmp", "--saida", f"{PASTA}/resultados/fluxo"]
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end="")
p.wait()
assert p.returncode == 0, "a medida falhou"

In [ ]:
# 4. Resultado e download
from IPython.display import Markdown, display
display(Markdown(open(f"{PASTA}/resultados/fluxo/fluxo.md").read()))
z = shutil.make_archive("/content/strata_fluxo", "zip", f"{PASTA}/resultados")
try:
    from google.colab import files
    files.download(z)
except Exception:
    print(z)